In [1]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model  # Not working properly so importing ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap

from langchain_groq import ChatGroq
 

In [8]:
loader = TextLoader("langchain_crewai_dataset.txt")
docs = loader.load()
docs

splitter = RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=50)
chunks = splitter.split_documents(docs)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_type="mmr",
                                     search_kwargs ={"k" : 4,
                                                     "lambda_mult" : 0.7
                                                     }
                                     )

In [10]:
import os 
from  dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
llm = init_chat_model("openai:o4-mini")
llm


ChatOpenAI(profile={'max_input_tokens': 200000, 'max_output_tokens': 100000, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000002CE35E734D0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002CE36540E60>, root_client=<openai.OpenAI object at 0x000002CE35BEEF60>, root_async_client=<openai.AsyncOpenAI object at 0x000002CE3604EB70>, model_name='o4-mini', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

In [ ]:
decomposition_prompt = PromptTemplate.from_template("""
You are an AI assistant. Decompose the following complex question into 4 smaller sub-questions for better document retrieval.
                                                   
Question : "{question}
                                                   
Sub-questions:
                                                   
""")

decomposition_chain = decomposition_prompt | llm | StrOutputParser()

In [18]:
query = "How does LangChain use memory and agents compared to CrewAI?"

decomposition_query = decomposition_chain.invoke({"question" : query})
print(decomposition_query)

Here are four targeted sub-questions you can use to guide your document retrieval:

1. What memory abstractions and storage mechanisms does LangChain provide, and how are they used in practice?  
2. How are agents designed and orchestrated within LangChain to carry out multi‐step tasks?  
3. What memory management strategies or modules does CrewAI implement in its workflows?  
4. How does CrewAI define and employ agents to perform and coordinate its tasks?


In [21]:
qa_prompt = PromptTemplate.from_template(""" 
Use the context below to answer the question.
                                         
Context : {context}
                                         
Question : {input}
                                         
""")

qa_chain = create_stuff_documents_chain(llm=llm,prompt=qa_prompt)

In [19]:
def full_query_decomposition_rag_pipeline(user_query):
    # Decompose the query
    sub_qs_text = decomposition_chain.invoke({"question": user_query})
    sub_questions = [q.strip("-•1234567890. ").strip() for q in sub_qs_text.split("\n") if q.strip()]
    
    results = []
    for subq in sub_questions:
        docs = retriever.invoke(subq)
        result = qa_chain.invoke({"input": subq, "context": docs})
        results.append(f"Q: {subq}\nA: {result}")
    
    return "\n\n".join(results)

In [ ]:
query = "How does LangChain use memory and agents compared to CrewAI?"
final_answer = full_query_decomposition_rag_pipeline(query)
print("✅ Final Answer:\n")
print(final_answer)

✅ Final Answer:

Q: Here are four focused sub-questions that will help retrieve the relevant documentation:
A: 1. Which section of the LangChain documentation describes the built-in tool wrappers (web search, calculators, code execution environments) and shows example usage?  
2. Where in the docs can I find guidance on adding and configuring custom API integrations as tools?  
3. Which pages cover LangChain’s prompt engineering features—templating syntax, input variables, formatting options, and reusability?  
4. Where is the prompt chaining and nesting guide, illustrating how to compose multiple prompt templates into complex workflows?

Q: What memory architectures and storage options does LangChain provide, and how are they used at runtime?
A: LangChain’s “memory” subsystem is really a set of interchangeable abstractions plus storage backends that let you keep track of what has happened in a conversation (or any chain-of-thought) and feed it back into prompts at runtime. Here’s a hi